In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 31.4187,
	"longitude": 73.0791,
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 31.458698272705078°N 73.11827850341797°E
Elevation: 192.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-07-21 00:00:00+00:00       30.549999
1   2026-07-21 01:00:00+00:00       29.200001
2   2026-07-21 02:00:00+00:00       28.950001
3   2026-07-21 03:00:00+00:00       28.000000
4   2026-07-21 04:00:00+00:00       28.299999
..                        ...             ...
163 2026-07-27 19:00:00+00:00       29.799999
164 2026-07-27 20:00:00+00:00       29.150000
165 2026-07-27 21:00:00+00:00       28.500000
166 2026-07-27 22:00:00+00:00       28.000000
167 2026-07-27 23:00:00+00:00       27.700001

[168 rows x 2 columns]


In [2]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# 1. Setup the client
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# 2. Air Quality Endpoint URL
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

# 3. Parameters for Faisalabad + Pollutants + Target AQI
aq_params = {
    "latitude": 31.4187,
    "longitude": 73.0791,
    "hourly": [
        "pm2_5", "pm10", "nitrogen_dioxide", 
        "sulphur_dioxide", "carbon_monoxide", 
        "ozone", "dust", "us_aqi"
    ],
    "past_days": 30,       # Fetch past 30 days for training/lag features
    "forecast_days": 5     # Fetch 5-day forecast horizon
}

responses = openmeteo.weather_api(aq_url, params=aq_params)
response = responses[0]

# 4. Extract hourly variables (Indices match the list order in `hourly` above)
hourly = response.Hourly()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),
    "pm2_5": hourly.Variables(0).ValuesAsNumpy(),
    "pm10": hourly.Variables(1).ValuesAsNumpy(),
    "nitrogen_dioxide": hourly.Variables(2).ValuesAsNumpy(),
    "sulphur_dioxide": hourly.Variables(3).ValuesAsNumpy(),
    "carbon_monoxide": hourly.Variables(4).ValuesAsNumpy(),
    "ozone": hourly.Variables(5).ValuesAsNumpy(),
    "dust": hourly.Variables(6).ValuesAsNumpy(),
    "us_aqi": hourly.Variables(7).ValuesAsNumpy(),
}

df_aq = pd.DataFrame(data=hourly_data)
print("Air Quality Data Shape:", df_aq.shape)
print(df_aq.head())

Air Quality Data Shape: (840, 9)
                       date      pm2_5        pm10  nitrogen_dioxide  \
0 2026-06-21 00:00:00+00:00  74.300003  132.699997         45.099998   
1 2026-06-21 01:00:00+00:00  70.099998  120.099998         37.000000   
2 2026-06-21 02:00:00+00:00  66.599998  114.800003         25.600000   
3 2026-06-21 03:00:00+00:00  59.799999  118.699997         16.000000   
4 2026-06-21 04:00:00+00:00  49.599998  110.400002          9.900000   

   sulphur_dioxide  carbon_monoxide  ozone   dust      us_aqi  
0              7.2            594.0   29.0  106.0  153.809525  
1              7.7            577.0   49.0  107.0  154.184525  
2              8.4            570.0   78.0  107.0  154.407745  
3              8.7            539.0  103.0  108.0  154.758926  
4              8.4            456.0  122.0  109.0  155.267853  
